# Colab: Treniranje EfficientNet modela

Ovaj notebook trenira **EfficientNet B0** model sa transfer learning-om na Google Colab-u.

**Preduslovi:** Pokrenite `colab_setup.ipynb` prvo!

## 1. Inicijalizacija okruzenja

In [ ]:
import os, sys, json

if os.path.exists('/content/colab_paths.json'):
    paths = json.load(open('/content/colab_paths.json'))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    paths = json.load(open('/content/drive/MyDrive/melanoma_results/colab_paths.json'))
    !pip install -q timm albumentations mahotas

sys.path.insert(0, paths['repo_dir'])

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NIJE DOSTUPAN'}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.config import Config
from src.data_utils import load_and_prepare_data
from src.training import run_cross_validation
from src.visualization import plot_all_folds_losses, plot_roc_curve, plot_fold_comparison

## 2. Konfiguracija

EfficientNet koristi transfer learning sa ImageNet tezinama. Sa GPU-om mozemo koristiti:
- Vece slike (224x224)
- Veci batch size (32)
- Vise epoha (20) sa 5-fold CV

In [ ]:
cfg = Config.colab()
cfg.model_type = 'efficientnet'
cfg.image_size = 224

# Postavi putanje iz setup-a
cfg.train_csv = paths['train_csv']
cfg.train_dir = paths['train_dir']
cfg.model_save_dir = paths['models_dir']

# Opciono: ogranici na subset
# cfg.subset_size = 1482  # Isti subset kao lokalno
# cfg.subset_size = None  # Ceo dataset (~33K slika)

print(f"Model: {cfg.model_type}")
print(f"Image size: {cfg.image_size}x{cfg.image_size}")
print(f"Epochs: {cfg.epochs}")
print(f"Folds: {cfg.num_folds}")
print(f"Batch size: {cfg.batch_size}")
print(f"Subset size: {cfg.subset_size}")
print(f"Device: {cfg.get_device()}")

## 3. Ucitavanje i priprema podataka

In [ ]:
df = load_and_prepare_data(cfg)

print(f"Ukupno slika: {len(df)}")
print(f"Raspodela klasa:")
print(f"  Benign:    {(df['target'] == 0).sum()} ({(df['target'] == 0).mean()*100:.1f}%)")
print(f"  Malignant: {(df['target'] == 1).sum()} ({(df['target'] == 1).mean()*100:.1f}%)")
print(f"\nBroj pacijenata: {df['patient_id'].nunique()}")
print(f"Feature dim: {cfg.feature_dim}")

## 4. Treniranje sa k-fold cross-validacijom

In [ ]:
effnet_results = run_cross_validation(df, cfg)

## 5. Vizualizacija rezultata

In [ ]:
fig = plot_all_folds_losses(effnet_results['per_fold_results'])
plt.show()

In [ ]:
fig = plot_roc_curve(
    effnet_results['oof_labels'],
    effnet_results['oof_probs'],
    fold_results=effnet_results['per_fold_results'],
    label='EfficientNet B0'
)
plt.show()

In [ ]:
fig = plot_fold_comparison(effnet_results['per_fold_results'])
plt.show()

## 6. Cuvanje rezultata na Google Drive

In [ ]:
import pickle

results_path = os.path.join(paths['results_dir'], 'effnet_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump({
        'oof_labels': effnet_results['oof_labels'],
        'oof_probs': effnet_results['oof_probs'],
        'oof_df': effnet_results['oof_df'],
        'mean_auc': effnet_results['mean_auc'],
        'std_auc': effnet_results['std_auc'],
        'per_fold_aucs': [r['best_val_auc'] for r in effnet_results['per_fold_results']],
    }, f)

print(f'Rezultati sacuvani u {results_path}')
print(f'Mean AUC: {effnet_results["mean_auc"]:.4f} +/- {effnet_results["std_auc"]:.4f}')

## Zakljucak

EfficientNet B0 model je treniran na GPU-u sa transfer learning-om. Rezultati su sacuvani na Google Drive-u.

Sledeci korak: pokrenite `colab_evaluation.ipynb` za poredjenje modela.